<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/kartik2006-del/efficient-finetuning-lora.git

fatal: destination path 'efficient-finetuning-lora' already exists and is not an empty directory.


In [4]:
%cd efficient-finetuning-lora

/content/efficient-finetuning-lora


In [5]:
!ls

lora_finetuning.ipynb  README.md


In [6]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned")
print(dataset)

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [7]:
print("Number of examples:", len(dataset["train"]))
print("Columns:", dataset["train"].column_names)

print("\nFirst example:")
print(dataset["train"][0])

Number of examples: 51760
Columns: ['output', 'input', 'instruction']

First example:
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [8]:
for column in dataset["train"].column_names:
    missing = sum(
        1 for x in dataset["train"][column]
        if x is None or x == ""
    )
    print(column, ":", missing)

output : 0
input : 32603
instruction : 0


In [9]:
train_data = dataset["train"]

with_input = sum(1 for x in train_data["input"] if x.strip() != "")
without_input = len(train_data) - with_input

print("With input:", with_input)
print("Without input:", without_input)

With input: 19157
Without input: 32603


In [10]:
import random

for i in random.sample(range(len(train_data)), 5):
    print("=" * 80)
    print("Instruction:", train_data[i]["instruction"])
    print("Input:", train_data[i]["input"])
    print("Output:", train_data[i]["output"])

Instruction: Search for word "ravioli" in the given sentence and make it bold.
Input: I had pizza, ravioli and salad for dinner last night.
Output: I had pizza, **ravioli** and salad for dinner last night.
Instruction: Research the benefits of yoga and explain them in a few sentences
Input: 
Output: Yoga is a practice with numerous physical, mental and spiritual benefits. Physically, yoga improves flexibility, strength, balance, and overall fitness. It also helps reduce stress, anxiety, and depression, as well as improve sleep quality and increase overall well-being. Mentally, yoga promotes mindfulness, concentration, and self-awareness, making it an excellent tool for stress management and self-improvement. Spiritually, many find that their yoga practice deepens their connection to a higher power or sense of purpose, helping them live more fulfilling lives. Overall, practicing yoga regularly can improve one's overall health and well-being, both body and mind.
Instruction: Classify the

In [11]:
examples = [
    (row["instruction"], row["input"], row["output"])
    for row in train_data
]
unique_examples = len(set(examples))

print("Total examples:", len(examples))
print("Unique examples:", unique_examples)
print("Duplicate examples:", len(examples) - unique_examples)

Total examples: 51760
Unique examples: 51756
Duplicate examples: 4


In [12]:
instructions = train_data["instruction"]

unique_instructions = len(set(instructions))

print("Total instructions:", len(instructions))
print("Unique instructions:", unique_instructions)
print("Duplicate instructions:", len(instructions) - unique_instructions)

Total instructions: 51760
Unique instructions: 51725
Duplicate instructions: 35


In [13]:
instruction_lengths = [
    len(x.split()) for x in train_data["instruction"]
]

output_lengths = [
    len(x.split()) for x in train_data["output"]
]

print("Instruction length:")
print("Minimum:", min(instruction_lengths))
print("Maximum:", max(instruction_lengths))
print("Average:", sum(instruction_lengths) / len(instruction_lengths))

print("\nOutput length:")
print("Minimum:", min(output_lengths))
print("Maximum:", max(output_lengths))
print("Average:", sum(output_lengths) / len(output_lengths))

Instruction length:
Minimum: 2
Maximum: 323
Average: 10.542716383307573

Output length:
Minimum: 1
Maximum: 507
Average: 109.9395285935085


In [14]:
max_instruction_index = instruction_lengths.index(max(instruction_lengths))

print("Longest instruction:")
print(train_data[max_instruction_index]["instruction"])

print("\nWord count:", instruction_lengths[max_instruction_index])

Longest instruction:
You are a smart assistant designed to help high school teachers come up with reading comprehension questions.
Given a piece of text, you must come up with a question and answer pair that can be used to test a student's reading comprehension abilities.
When coming up with this question/answer pair, you must respond in the following format:
```
{{
    "question": "$YOUR_QUESTION_HERE",
    "answer": "$THE_ANSWER_HERE"
}}
```
Everything between the ``` must be valid json.
Please come up with a question/answer pair, in the specified JSON format, for the following text:
----------------
Previously, the use of dye-sub printing was limited to industrial or high-end commercial printing. Dye-sub photo printing has been used in medical imaging, graphic arts proofing, security, and broadcast-related applications. Today, it is extremely popular in event photography and photo booths or kiosks that require high-speed, on-demand printing.

Alps Electric produced the first quality

In [15]:
max_output_index = output_lengths.index(max(output_lengths))

print("Longest output:")
print(train_data[max_output_index]["output"])

print("\nWord count:", output_lengths[max_output_index])

Longest output:
Here is an implementation of a red-black tree in C++:

```
#include <iostream>

using namespace std;

enum Color {RED, BLACK};

struct Node {
    int val;
    Color color;
    Node *left, *right, *parent;
    
    Node() {}
    
    Node(int v, Color c) {
        val = v;
        color = c;
        left = right = parent = NULL;
    }
};

class RedBlackTree {
    private:
        Node *root;
        
    public:
        RedBlackTree() { root = NULL; }
        
        /* Helper functions */
        Node* grandparent(Node *n) {
            if (n && n->parent) { return n->parent->parent; }
            else { return NULL; }
        }
        
        Node* uncle(Node *n) {
            Node *g = grandparent(n);
            if (!g) { return NULL; }
            if (n->parent == g->left) { return g->right; }
            else { return g->left; }
        }
        
        Node* sibling(Node *n) {
            if (!n || !n->parent) { return NULL; }
            if (n == n->parent->

In [16]:
df = train_data.to_pandas()

print("Before removing duplicates:", len(df))

df = df.drop_duplicates(
    subset=["instruction", "input", "output"]).reset_index(drop=True)

print("After removing duplicates:", len(df))
print("Removed:", 51760 - len(df))

Before removing duplicates: 51760
After removing duplicates: 51756
Removed: 4


In [17]:
def format_example(row):
    if row["input"].strip():
        user_content = row["instruction"] + "\n\nInput:\n" + row["input"]
    else:
        user_content = row["instruction"]

    return {
        "messages": [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": row["output"]
            }
        ]
    }

In [18]:
from datasets import Dataset

formatted_data = Dataset.from_pandas(df)

formatted_data = formatted_data.map(
    format_example,
    remove_columns=["instruction", "input", "output"]
)

Map:   0%|          | 0/51756 [00:00<?, ? examples/s]

In [19]:
print(formatted_data[0])

{'messages': [{'content': 'Give three tips for staying healthy.', 'role': 'user'}, {'content': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'role': 'assistant'}]}


In [20]:
train_test = formatted_data.train_test_split(
    test_size=0.20,
    seed=42
)

train_data = train_test["train"]

temp_data = train_test["test"].train_test_split(
    test_size=0.50,
    seed=42
)

validation_data = temp_data["train"]
test_data = temp_data["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 41404
Validation: 5176
Test: 5176


In [21]:
print(train_data[0])
print("\nTrain:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))
print(train_data.column_names)

{'messages': [{'content': 'Imagine you are advising a fictional character who seeks to purchase a suitable spaceship for interstellar travel. Recommend three critical factors they should consider and explain why each is important.', 'role': 'user'}, {'content': "1. Fuel efficiency - To maximize reach and minimize fuel costs, it's essential to choose a ship with excellent fuel efficiency.\\n2. Advanced navigation system - Accurate navigation is crucial for avoiding hazards and reaching intended destinations in the vastness of space.\\n3. Maintenance and repair capabilities - The ship should be easy to maintain and repair, as it may become necessary during long journeys with limited access to external services.", 'role': 'assistant'}]}

Train: 41404
Validation: 5176
Test: 5176
['messages']


In [23]:
!pip install -q transformers accelerate

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [25]:
prompt = "Explain machine learning in simple words."

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True)

print(response)

Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time without being explicitly programmed. It involves using algorithms to analyze patterns in large amounts of data and make predictions or decisions based on those patterns.

In simpler terms, imagine you have a bunch of toys scattered around your room. Machine learning can help you organize these toys into different categories, find the ones you like best, and even suggest new toys for you to play with if you're not sure which one
